In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

print(tf.__version__)
print(tf.config.list_physical_devices())


# CNN FOR DATE : DATA , SPECTRAL IMAGE


In [ ]:
with tf.device('/GPU:0'):
    X_raw = np.array([
        # 0: STABLE
        [5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],
        [2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],

        # 1: LARGE WAVE
        [1,9,1,9,1,9,1,9,1,9,1,9,1,9,1,9,1,9,1,9],
        [2,8,2,8,2,8,2,8,2,8,2,8,2,8,2,8,2,8,2,8],

        # 2: SMALL WAVE
        [5.0,5.2,5.0,5.2,5.0,5.2,5.0,5.2,5.0,5.2,5.0,5.2,5.0,5.2,5.0,5.2,5.0,5.2,5.0,5.2],
        [4.0,4.2,4.0,4.2,4.0,4.2,4.0,4.2,4.0,4.2,4.0,4.2,4.0,4.2,4.0,4.2,4.0,4.2,4.0,4.2],

        # 3: RISING
        [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],
        [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24],

        # 4: FALLING
        [20,19,18,17,16,15,14,13,12,11,10,9,8,7,6,5,4,3,2,1],
        [25,24,23,22,21,20,19,18,17,16,15,14,13,12,11,10,9,8,7,6],

        # 5: SPIKY
        [1,1,9,1,1,1,1,8,1,1,1,1,9,1,1,1,1,8,1,1],
        [1,8,1,1,1,1,1,1,9,1,1,1,1,1,1,1,8,1,1,1],

        # 6: PEAK
        [1,1,1,2,3,5,8,10,10,8,5,3,2,1,1,1,1,1,1,1],
        [0,0,1,2,4,6,9,9,9,6,4,2,1,0,0,0,0,0,0,0],

        # 7: VALLEY
        [9,9,9,7,5,3,1,0,0,1,3,5,7,9,9,9,9,9,9,9],
        [10,10,8,6,4,2,1,1,2,4,6,8,10,10,10,10,10,10,10,10]
    ])

    X = X_raw.reshape((16, 20, 1))
    y = np.array([0,0, 1,1, 2,2, 3,3, 4,4, 5,5, 6,6, 7,7])
    model = tf.keras.models.Sequential([
        # We use kernel_size=3 to look at small chunks of the 20-step sequence
        tf.keras.layers.Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(20, 1)),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        # Adding a second Conv layer helps the model distinguish between complex waves
        tf.keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu'),
        tf.keras.layers.Conv1D(filters=128, kernel_size=3, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Flatten(),

        # Output layer MUST be 8 for your 8 pattern classes
        tf.keras.layers.Dense(8, activation='softmax')
    ])

    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    # Training for 1000 epochs to ensure it learns the subtle differences (like Small Wave)
    model.fit(X, y, epochs=1000, verbose=1)
    model.save('cnn_8classes.h5')

In [ ]:
mod = tf.keras.models.load_model('cnn_8classes.h5')

class_names = ["Stable", "Large Wave", "Small Wave", "Rising", "Falling", "Spiky", "Peak", "Valley"]

sample_data = [2, 2, 2.1, 2, 2, 2, 2.2, 2, 2, 2, 2, 9.5, 2, 2, 2, 2.1, 2, 2, 2, 2]
new_sample = np.array([sample_data]).reshape(1, 20, 1)

pred = mod.predict(new_sample)

predicted_index = np.argmax(pred)
confidence = pred[0][predicted_index] * 100

print(f"Prediction probabilities: {pred}")
print(f"---")
print(f"Result: {class_names[predicted_index]} Pattern Detected!")
print(f"Confidence: {confidence:.2f}%")


# CNN + TRANSFORMERS HYBRID


In [ ]:
with tf.device('/GPU:0'):
    # --- DATA PREPARATION (Same as yours) ---
    X_raw = np.array([
        [5]*20, [2]*20, # 0: Stable
        [1,9]*10, [2,8]*10, # 1: Large Wave
        [5.0,5.2]*10, [4.0,4.2]*10, # 2: Small Wave
        list(range(1,21)), list(range(5,25)), # 3: Rising
        list(range(20,0,-1)), list(range(25,5,-1)), # 4: Falling
        [1,1,9,1,1,1,1,8,1,1,1,1,9,1,1,1,1,8,1,1], [1,8,1,1,1,1,1,1,9,1,1,1,1,1,1,1,8,1,1,1], # 5: Spiky
        [1,1,1,2,3,5,8,10,10,8,5,3,2,1,1,1,1,1,1,1], [0,0,1,2,4,6,9,9,9,6,4,2,1,0,0,0,0,0,0,0], # 6: Peak
        [9,9,9,7,5,3,1,0,0,1,3,5,7,9,9,9,9,9,9,9], [10,10,8,6,4,2,1,1,2,4,6,8,10,10,10,10,10,10,10,10] # 7: Valley
    ])
    STEPS=20
    X = X_raw.reshape((16, STEPS, 1))
    y = np.array([0,0, 1,1, 2,2, 3,3, 4,4, 5,5, 6,6, 7,7])


    inputs=tf.keras.layers.Input(shape=(STEPS, 1))
    x=tf.keras.layers.Conv1D(filters=16, kernel_size=3, activation='relu')(inputs)
    x=tf.keras.layers.MaxPooling1D(pool_size=2)(x)
    x=tf.keras.layers.Conv1D(filters=32, kernel_size=3, activation='relu')(x)
    attention=tf.keras.layers.MultiHeadAttention(num_heads=4,key_dim=32)(x,x)

    x=tf.keras.layers.add([x,attention])
    x=tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)

    x=tf.keras.layers.Flatten()(x)
    x=tf.keras.layers.Dense(64, activation='relu')(x)
    x=tf.keras.layers.Dropout(0.1)(x)
    output=tf.keras.layers.Dense(8,activation='softmax')(x)

    model=tf.keras.models.Model(inputs=inputs, outputs=output)

    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.fit(X, y, epochs=1000, verbose=1)
    model.save('cnn_trans_8classes.h5')

In [ ]:
mod = tf.keras.models.load_model('cnn_trans_8classes.h5')

class_names = ["Stable", "Large Wave", "Small Wave", "Rising", "Falling", "Spiky", "Peak", "Valley"]

sample_data = [0.5, 0.6, 0.8, 1.2, 2.5, 4.8, 7.2, 9.5, 10.2, 9.8, 7.5, 5.1, 3.2, 1.8, 1.1, 0.8, 0.6, 0.5, 0.4, 0.4]
new_sample = np.array([sample_data]).reshape(1, 20, 1)

pred = mod.predict(new_sample)

predicted_index = np.argmax(pred)
confidence = pred[0][predicted_index] * 100

print(f"Prediction probabilities: {pred}")
print(f"---")
print(f"Result: {class_names[predicted_index]} Pattern Detected!")
print(f"Confidence: {confidence:.2f}%")


# CNN + TRANSFORMERS USING EXTERNAL DATASET



## Getting data


In [ ]:
df = pd.read_csv('applw.csv')

frequencies = df.columns[2:].astype(float).values
X_raw = df.iloc[:, 2:].values
y_raw = df['Dry matter'].values

y_scaled = y_raw * 100.0
STEPS = X_raw.shape[1]

X_reshaped = X_raw.reshape((X_raw.shape[0], STEPS, 1))

X_train = X_reshaped[:-10]
y_train = y_scaled[:-10]

X_test = X_reshaped[-10:]
y_test = y_scaled[-10:]

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train.reshape(-1, STEPS)).reshape(-1, STEPS, 1)

X_test_scaled = scaler.transform(X_test.reshape(-1, STEPS)).reshape(-1, STEPS, 1)

print(f"Training data shape: {X_train_scaled.shape}")
print(f"Test data shape: {X_test_scaled.shape}")

In [ ]:
np.mean(y_train)

## Training Model ##

In [ ]:
with tf.device('/GPU:0'):
    inputs = tf.keras.layers.Input(shape=(STEPS, 1))

    layer = tf.keras.layers.Conv1D(filters=32, kernel_size=5, activation='swish', padding='same')(inputs)
    layer = tf.keras.layers.MaxPooling1D(pool_size=2)(layer)
    layer = tf.keras.layers.Conv1D(filters=64, kernel_size=5, activation='swish', padding='same')(layer)

    attention = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=64)(layer, layer)
    layer = tf.keras.layers.add([layer, attention])

    layer = tf.keras.layers.Flatten()(layer)
    layer = tf.keras.layers.Dense(128, activation='swish',kernel_regularizer=tf.keras.regularizers.l2(0.001))(layer)

    # RESET: Back to 0.1 Dropout (The sweet spot for your dataset)
    layer = tf.keras.layers.Dropout(0.1)(layer)

    bias = tf.keras.initializers.Constant(np.mean(y_train))
    output = tf.keras.layers.Dense(1, activation='linear', bias_initializer=bias)(layer)

    model = tf.keras.models.Model(inputs=inputs, outputs=output)
    model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae'],weighted_metrics=[])

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=30, min_lr=0.00001, verbose=1)

    # RESET: Early Stopping is back on to prevent it from destroying itself
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)
    weights = np.ones(len(y_train))

    # 2. Find the extreme apples (below 14.5 and above 16.5)
    extreme_apples_mask = (y_train < 14.5) | (y_train > 16.5)

    # 3. Force the model to pay 5x more attention to these rare apples!
    weights[extreme_apples_mask] = 5.0

    print("--- STARTING TRAINING (BASELINE RESTORED) ---")
    history = model.fit(
        X_train_scaled,
        y_train,
        epochs=1000, # Safe to leave high because Early Stopping will catch it
        callbacks=[],
        validation_split=0.2,
        sample_weight=weights,
        verbose=1
    )

    # Let's save this as your "safe" model
    model.save('apple.h5')

## Manual Input ##

In [ ]:
mod = tf.keras.models.load_model('apple.h5')

sample_data = [0.033587163,0.034039606,0.034473633,0.037227483,0.040992484,0.044824456,0.048763599,0.052703239,0.051158329,0.048139377,0.046159542,0.0452295,0.044297007,0.042951248,0.041553705,0.040640028,0.040172554,0.039705373,0.03893958,0.03823347,0.038037827,0.038748962,0.039459964,0.04058425,0.04314863,0.045712212,0.048952447,0.052306679,0.054678147,0.054673636,0.056135207,0.063992677,0.071851223,0.07192594,0.068357851,0.074559967,0.088742063,0.103846869,0.125765601,0.147685005,0.169977164,0.192432397,0.214890437,0.243621094,0.273679638,0.302510667,0.327288397,0.352069331,0.379691511,0.407847866,0.429857316,0.439659672,0.448900258,0.453127475,0.457319888,0.457767669,0.452937534,0.448107079,0.4334197,0.415163083,0.398154228,0.407117863,0.4176672,0.464462245,0.53262744,0.596172248,0.651958919,0.706377834,0.730139416,0.751973428,0.766034026,0.776493949,0.786955006,0.78977976,0.788517049,0.787257498,0.785270821,0.783174675,0.781278478,0.779594307,0.778017757,0.777393569,0.77679034,0.776971983,0.778351632,0.779698902,0.780488822,0.781277465,0.78140121,0.781319889,0.784083508,0.786884068,0.789652873,0.791104548,0.792447429,0.793775593,0.794675338,0.795533524,0.792747199,0.788529894,0.784488074,0.780946018,0.777404528,0.775109627,0.774923732,0.774551964,0.772531894,0.770512432,0.771631101,0.774420464,0.776395835,0.775775107,0.774085404,0.763965633,0.75350955,0.741530111,0.727443672,0.719889067,0.719128884,0.718368572,0.715656011,0.707887123,0.69902045,0.689068804,0.679116865,0.669111522,0.659170071,0.646854923,0.632931905,0.619009602,0.605258186,0.593932156,0.582771513,0.572218165,0.561880077,0.553803899,0.553528739,0.553253967,0.552748295,0.548648012,0.544424519
]
new_sample = np.array([sample_data]).reshape(1, len(sample_data), 1)

pred = mod.predict(new_sample)

prediction=pred[0][0]
mean=np.mean(y)
print(f"Prediction probabilities: {pred}")
print(f"---")
print(f"Result: {prediction} Pattern Detected! needed 0.1743")

## BLIND TESTING ##

In [ ]:
predictions = model.predict(X_test_scaled)

for i in range(len(y_test)):
    # Divide both true and predicted values by 100 to get back to the original decimal
    true_val = y_test[i] / 100.0
    pred_val = predictions[i][0] / 100.0

    error = abs(true_val - pred_val)

    print(f"Apple {i+1} | True: {true_val:.4f} | Predicted: {pred_val:.4f} | Error: {error:.4f}")

In [ ]:
predictions = model.predict(X_test_scaled)
total_accuracy = 0

for i in range(len(y_test)):
    true_val = y_test[i] / 100.0
    pred_val = predictions[i][0] / 100.0
    error = abs(true_val - pred_val)

    percentage_error = (error / true_val) * 100
    accuracy = 100.0 - percentage_error

    total_accuracy += accuracy

    print(f"Apple {i+1} | True: {true_val:.4f} | Predicted: {pred_val:.4f} | Error: {error:.4f} | Accuracy: {accuracy:.2f}%")

avg_accuracy = total_accuracy / len(y_test)
print(f"\n=> AVERAGE BLIND TEST ACCURACY: {avg_accuracy:.2f}%")

In [ ]:
print("\n--- RESULTS FOR ALL DATA (TRAIN + TEST) ---")

from sklearn.preprocessing import MinMaxScaler

# 1. Re-initialize and explicitly fit the scaler on the training data
# (This permanently prevents the NotFittedError)
scaler = MinMaxScaler()
X_train_only = X[:-10]  # The data the model actually trained on
scaler.fit(X_train_only.reshape(-1, STEPS)) # The scaler learns the rules here!

# 2. Scale the ENTIRE dataset using those rules
X_all_scaled = scaler.transform(X.reshape(-1, STEPS)).reshape(-1, STEPS, 1)

# 3. Make predictions for every single apple
all_predictions = model.predict(X_all_scaled)

total_accuracy = 0

# 4. Loop through the entire target array
for i in range(len(y)):
    # Divide both true and predicted values by 100 to get back to the original decimal
    true_val = y[i] / 100.0
    pred_val = all_predictions[i][0] / 100.0

    error = abs(true_val - pred_val)

    percentage_error = (error / true_val) * 100
    accuracy = 100.0 - percentage_error

    total_accuracy += accuracy

    print(f"Apple {i+1} | True: {true_val:.4f} | Predicted: {pred_val:.4f} | Error: {error:.4f} | Accuracy: {accuracy:.2f}%")

# Calculate and print the overall average accuracy for the entire dataset
avg_accuracy = total_accuracy / len(y)
print(f"\n=> AVERAGE OVERALL ACCURACY (TRAIN + TEST): {avg_accuracy:.2f}%")


# GEN


In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# GEN: dry-matter regression from apple spectroscopy data.
# This is written sequentially on purpose: no custom classes, no custom functions.
RANDOM_STATE = 42
TEST_SIZE = 10
CSV_FILE = 'applw.csv'
TARGET_COL = 'Dry matter'
ID_COL = 'Apple'

# --------------------
# Load data
# --------------------
df = pd.read_csv(CSV_FILE)
wavelength_cols = df.columns[2:]
frequencies = wavelength_cols.astype(float).values

X_raw = df[wavelength_cols].to_numpy(dtype=np.float64)
y = df[TARGET_COL].to_numpy(dtype=np.float64)

X_train = X_raw[:-TEST_SIZE]
y_train = y[:-TEST_SIZE]
X_test = X_raw[-TEST_SIZE:]
y_test = y[-TEST_SIZE:]

print(f"Loaded {CSV_FILE}: {X_raw.shape[0]} apples x {X_raw.shape[1]} wavelengths")
print(f"Predicting: {TARGET_COL}")
print(f"Train rows: {len(X_train)} | Blind test rows: {len(X_test)}")


In [ ]:
# --------------------
# Build spectroscopy features for TRAIN
# --------------------
train_mean = X_train.mean(axis=1, keepdims=True)
train_std = X_train.std(axis=1, keepdims=True) + 1e-12
train_snv = (X_train - train_mean) / train_std
train_first_derivative = np.diff(train_snv, axis=1)
train_second_derivative = np.diff(train_first_derivative, axis=1)
train_stats = np.column_stack([
    X_train.mean(axis=1),
    X_train.std(axis=1),
    X_train.min(axis=1),
    X_train.max(axis=1),
    X_train[:, -1] - X_train[:, 0],
    np.trapz(X_train, axis=1),
])
X_train_features = np.hstack([
    X_train,
    train_snv,
    train_first_derivative,
    train_second_derivative,
    train_stats,
])

# --------------------
# Build spectroscopy features for BLIND TEST
# --------------------
test_mean = X_test.mean(axis=1, keepdims=True)
test_std = X_test.std(axis=1, keepdims=True) + 1e-12
test_snv = (X_test - test_mean) / test_std
test_first_derivative = np.diff(test_snv, axis=1)
test_second_derivative = np.diff(test_first_derivative, axis=1)
test_stats = np.column_stack([
    X_test.mean(axis=1),
    X_test.std(axis=1),
    X_test.min(axis=1),
    X_test.max(axis=1),
    X_test[:, -1] - X_test[:, 0],
    np.trapz(X_test, axis=1),
])
X_test_features = np.hstack([
    X_test,
    test_snv,
    test_first_derivative,
    test_second_derivative,
    test_stats,
])

# --------------------
# Build spectroscopy features for ALL DATA
# --------------------
all_mean = X_raw.mean(axis=1, keepdims=True)
all_std = X_raw.std(axis=1, keepdims=True) + 1e-12
all_snv = (X_raw - all_mean) / all_std
all_first_derivative = np.diff(all_snv, axis=1)
all_second_derivative = np.diff(all_first_derivative, axis=1)
all_stats = np.column_stack([
    X_raw.mean(axis=1),
    X_raw.std(axis=1),
    X_raw.min(axis=1),
    X_raw.max(axis=1),
    X_raw[:, -1] - X_raw[:, 0],
    np.trapz(X_raw, axis=1),
])
X_all_features = np.hstack([
    X_raw,
    all_snv,
    all_first_derivative,
    all_second_derivative,
    all_stats,
])

print(f"Feature shape after preprocessing: {X_train_features.shape[1]} columns")


In [ ]:
# --------------------
# Train model 1: PLS Regression
# --------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled = scaler.transform(X_test_features)
X_all_scaled = scaler.transform(X_all_features)

pls_model = PLSRegression(n_components=8)
pls_model.fit(X_train_scaled, y_train)

pls_train_predictions = pls_model.predict(X_train_scaled).ravel()
pls_test_predictions = pls_model.predict(X_test_scaled).ravel()
pls_all_predictions = pls_model.predict(X_all_scaled).ravel()

# --------------------
# Train model 2: ExtraTrees Regression
# --------------------
extra_trees_model = ExtraTreesRegressor(
    n_estimators=1000,
    random_state=RANDOM_STATE,
    max_features=0.70,
    min_samples_leaf=1,
    n_jobs=-1,
)
extra_trees_model.fit(X_train_features, y_train)

extra_train_predictions = extra_trees_model.predict(X_train_features)
extra_test_predictions = extra_trees_model.predict(X_test_features)
extra_all_predictions = extra_trees_model.predict(X_all_features)

# --------------------
# Blend both models
# --------------------
pls_weight = 0.48
extra_trees_weight = 0.52

train_predictions = (pls_weight * pls_train_predictions) + (extra_trees_weight * extra_train_predictions)
test_predictions = (pls_weight * pls_test_predictions) + (extra_trees_weight * extra_test_predictions)
all_predictions = (pls_weight * pls_all_predictions) + (extra_trees_weight * extra_all_predictions)

joblib.dump({
    'scaler': scaler,
    'pls_model': pls_model,
    'extra_trees_model': extra_trees_model,
    'pls_weight': pls_weight,
    'extra_trees_weight': extra_trees_weight,
    'wavelength_cols': list(wavelength_cols),
}, 'apple_gen_dry_matter_model.pkl')
print('Saved model parts: apple_gen_dry_matter_model.pkl')

# --------------------
# Results
# --------------------
train_accuracy = np.mean(100.0 - (np.abs(y_train - train_predictions) / np.abs(y_train) * 100.0))
test_accuracy = np.mean(100.0 - (np.abs(y_test - test_predictions) / np.abs(y_test) * 100.0))
all_accuracy = np.mean(100.0 - (np.abs(y - all_predictions) / np.abs(y) * 100.0))

print("\n--- TRAIN ---")
print(f"Accuracy: {train_accuracy:.2f}%")
print(f"MAE:      {mean_absolute_error(y_train, train_predictions):.6f}")
print(f"R2:       {r2_score(y_train, train_predictions):.4f}")

print("\n--- BLIND TEST - LAST 10 APPLES ---")
print(f"Accuracy: {test_accuracy:.2f}%")
print(f"MAE:      {mean_absolute_error(y_test, test_predictions):.6f}")
print(f"R2:       {r2_score(y_test, test_predictions):.4f}")

print("\n--- ALL DATA ---")
print(f"Accuracy: {all_accuracy:.2f}%")
print(f"MAE:      {mean_absolute_error(y, all_predictions):.6f}")
print(f"R2:       {r2_score(y, all_predictions):.4f}")

print("\n--- BLIND TEST DETAILS ---")
for apple_id, true_val, pred_val in zip(df[ID_COL].iloc[-TEST_SIZE:], y_test, test_predictions):
    error = abs(true_val - pred_val)
    row_accuracy = 100.0 - (error / abs(true_val) * 100.0)
    print(
        f"{apple_id} | True: {true_val:.6f} | Predicted: {pred_val:.6f} | "
        f"Error: {error:.6f} | Accuracy: {row_accuracy:.2f}%"
    )

# For manual prediction later, repeat the same preprocessing on one new sample,
# then blend PLS and ExtraTrees predictions with the same weights.
